In [1]:
import torch
import json
import PIL.Image
import polars as pl
import numpy as np
from bioclip import TreeOfLifeClassifier, Rank
from bioclip.predict import create_classification_dict

/Users/jpb67/Documents/work/pybioclip/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
taxa_path = "taxa.csv"
taxon_rank = "order" # Change this to "species" to target just the species in your CSV

# Paths to save filtered list of embeddings/labels
image_embeddings_path = "image_embeddings.npy"
embedding_labels_path = "embedding_labels.json"

## Read CSV to create a list of taxon values

In [3]:
print("Reading", taxa_path, "extracting", taxon_rank, "values.")
df = pl.read_csv(taxa_path)
target_values = set(pl.Series(df.select(taxon_rank).drop_nulls()).str.to_lowercase().unique().to_list())
print("Found", len(target_values), taxon_rank, "values.")

Reading taxa.csv extracting order values.
Found 25 order values.


## Load TOL classifier

In [4]:
print("Loading TOL classifier")
classifier = TreeOfLifeClassifier()
print("TOL: number of labels:", len(classifier.txt_names))
print("TOL: image embeddings shape:", classifier.txt_features.shape)

Loading TOL classifier
TOL: number of labels: 384490
TOL: image embeddings shape: torch.Size([512, 384490])


## Filter to the target taxon setting

In [5]:
print("Finding embeddings matching the targets.")
found_items = []
for i, txt_name in enumerate(classifier.txt_names):
    name_dict = create_classification_dict(txt_name, Rank.SPECIES)
    if name_dict[taxon_rank].lower() in target_values:
        found_items.append((i, txt_name))

print("Found", len(found_items), "embeddings matching the", taxon_rank, "values")

Finding embeddings matching the targets.
Found 152300 embeddings matching the order values


## Create embedding tensor for filtered embeddings

In [6]:
print("Building the image embedding tensor")
txt_feature_ary = []
new_txt_names = []
for i, txt_name in found_items:
    txt_feature_ary.append(classifier.txt_features[:,i])
    new_txt_names.append(txt_name)
new_txt_feature = torch.stack(txt_feature_ary, dim=1)

Building the image embedding tensor


## Save image embeddings and labels

In [7]:
print("Saving the image embedding tensor as ", image_embeddings_path)
np.save(image_embeddings_path, new_txt_feature)

print("Saving the embedding labels as ", embedding_labels_path)
with open(embedding_labels_path, "w") as outfile:
    json.dump(new_txt_names, outfile)

print("Done")

Saving the image embedding tensor as  image_embeddings.npy
Saving the embedding labels as  embedding_labels.json
Done


## Load classifier with custom embeddings/labels

In [8]:
classifier = TreeOfLifeClassifier()
print("Loading custom embeddings", embedding_labels_path, "and", image_embeddings_path)
with open(embedding_labels_path) as infile:
    classifier.txt_names = json.load(infile)
classifier.txt_features = torch.from_numpy(np.load(image_embeddings_path))

for pred in classifier.predict('bf.png', Rank.ORDER):
    print(pred)

Loading custom embeddings embedding_labels.json and image_embeddings.npy
{'file_name': 'bf.png', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Lepidoptera', 'score': 0.9992584586143494}
{'file_name': 'bf.png', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Coleoptera', 'score': 0.00045276302262209356}
{'file_name': 'bf.png', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Plecoptera', 'score': 0.00015735262422822416}
{'file_name': 'bf.png', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Odonata', 'score': 4.7870242269709706e-05}
{'file_name': 'bf.png', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Hymenoptera', 'score': 3.739578460226767e-05}


### PIL version of prediction

In [9]:
img = PIL.Image.open('bf.png').convert("RGB")
images = [img]

img_features = classifier.create_image_features(images)
for probs in classifier.create_probabilities(img_features, classifier.txt_features):
    for pred in classifier.format_grouped_probs("", probs, rank=Rank.ORDER, min_prob=1e-9, k=5):
        print(pred)

{'file_name': '', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Lepidoptera', 'score': 0.9992584586143494}
{'file_name': '', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Coleoptera', 'score': 0.00045276302262209356}
{'file_name': '', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Plecoptera', 'score': 0.00015735262422822416}
{'file_name': '', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Odonata', 'score': 4.7870242269709706e-05}
{'file_name': '', 'kingdom': 'Animalia', 'phylum': 'Arthropoda', 'class': 'Insecta', 'order': 'Hymenoptera', 'score': 3.739578460226767e-05}
